# Projet 4 — Partie 3 : Étude du déploiement sur AWS

**DataSoluTech — étude technique et recommandation d’architecture**  
**Date de consultation des sources : 5 novembre 2026**

## 1. Contexte et objectif

Le projet DataSoluTech consiste à migrer un dataset médical CSV vers MongoDB et à étudier une solution capable d'évoluer si la volumétrie et le nombre d'utilisateurs augmentent.

Les deux premières parties ont permis de :

- analyser **55 500 lignes / 15 colonnes** et supprimer **534 doublons exacts** selon la règle de nettoyage retenue ;
- convertir les **54 966 enregistrements** conservés en documents MongoDB structurés en quatre sous-documents (`patient`, `medical`, `hospitalisation`, `facturation`) ;
- automatiser l'importation en Python et tester son intégrité ;
- conteneuriser le script Python et MongoDB avec Docker Compose ;
- configurer trois comptes (`admin`, `analyste`, `data_engineer`) et automatiser leur création pour une installation MongoDB neuve.

**Objectif de cette troisième partie :** comparer des services AWS, puis proposer une architecture justifiée. Il s'agit d'une **étude**, et **non d'un déploiement AWS réalisé**.

**Précaution :** le petit fichier d'exercice ne démontre pas, à lui seul, un besoin de sharding ou de grande capacité. Pour dimensionner la production, il faudra connaître les volumes futurs, le rythme des écritures, les requêtes, la disponibilité attendue et le budget.

## 2. Besoins du client et critères de choix

| Besoin | Ce qu'il faut prévoir |
|---|---|
| Scalabilité | Pouvoir augmenter les capacités de traitement et de stockage lorsque la charge augmente |
| Sécurité | Chiffrement, comptes limités aux droits nécessaires, accès réseau privé, gestion des secrets |
| Disponibilité | Tolérance aux pannes, surveillance et solution de reprise |
| Fiabilité des données | Vérifications après migration, sauvegardes et tests de restauration |
| Portabilité | Réutiliser autant que possible notre script Python et son image Docker |
| Maîtrise des coûts | Dimensionnement selon la charge réelle et suivi de la facturation |

**Deux types de mise à l'échelle :**

- **Verticale :** augmenter les ressources d'un serveur (CPU, mémoire).
- **Horizontale :** ajouter des instances ou répartir les données et requêtes entre plusieurs machines.

Docker facilite le déploiement, mais **ne rend pas MongoDB horizontalement scalable à lui seul**. La réplication améliore d'abord la disponibilité ; le *sharding* répartit les données entre plusieurs partitions pour soutenir certaines montées en charge.

## 3. Amazon S3 — Stockage des fichiers

**Fonctionnement.** Amazon Simple Storage Service (S3) est un service de **stockage d'objets** organisés dans des compartiments appelés *buckets*. Il convient au stockage de fichiers et non au remplacement direct d'une collection MongoDB.

**Utilité pour DataSoluTech :**

- conserver le CSV d'origine et les fichiers d'entrée d'un pipeline ;
- archiver les rapports de contrôle, les exports et les fichiers de sauvegarde produits par une procédure dédiée ;
- définir des règles de cycle de vie afin de déplacer les anciennes archives vers des classes de stockage moins coûteuses.

**Avantages :** stockage évolutif, classes de stockage adaptées à la fréquence d'accès, chiffrement au repos activé par défaut, intégration avec les services AWS.

**Limites / vigilance :** S3 n'exécute pas de requêtes MongoDB ; la facturation dépend du stockage, des requêtes et des transferts. Déposer un CSV sur S3 **ne constitue pas une sauvegarde complète d'une base MongoDB en exploitation**. Les buckets devront rester privés, avec des règles IAM strictes, des politiques de conservation et, si nécessaire, des clés KMS.

**Application proposée :** un bucket privé pour les fichiers d'entrée et éventuellement un autre pour les exports/archives. L'image Docker Python devra être adaptée pour lire les données depuis S3 ou les télécharger avant la migration : notre version locale lit actuellement un fichier dans `data/`.

**Sources :** [Présentation d'Amazon S3](https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html) · [Classes de stockage](https://docs.aws.amazon.com/AmazonS3/latest/userguide/storage-class-intro.html) · [Chiffrement par défaut](https://docs.aws.amazon.com/AmazonS3/latest/userguide/default-encryption-faq.html).

## 4. Amazon RDS — Bases de données relationnelles

**Fonctionnement.** Amazon Relational Database Service (RDS) est un service managé pour les bases **relationnelles SQL**, par exemple PostgreSQL, MySQL, MariaDB, SQL Server, Oracle ou Db2. AWS gère notamment une partie des tâches d'infrastructure, des sauvegardes et de la maintenance.

**Avantages :** solution pertinente pour des applications nécessitant des jointures et des contraintes relationnelles, gestion simplifiée de l'administration, possibilités de haute disponibilité et de sauvegarde.

**Limite décisive :** **il n'existe pas de moteur « RDS pour MongoDB »**. Utiliser RDS demanderait de reconcevoir notre modèle documentaire en modèle relationnel et de modifier notre migration.

**Décision pour le projet :** RDS est intéressant à connaître et à comparer, mais **n'est pas retenu pour héberger la base MongoDB**. Il pourrait servir à d'autres applications relationnelles de l'entreprise.

**Source :** [Fonctionnalités et moteurs d'Amazon RDS](https://aws.amazon.com/rds/features/).

## 5. Amazon DocumentDB — Base documentaire managée

**Fonctionnement.** Amazon DocumentDB est un service de base de données documentaire **compatible avec certaines API MongoDB**, géré par AWS. Il prend notamment en charge la compatibilité avec les API MongoDB 8.0.

**Avantages pour DataSoluTech :**

- stocker les documents imbriqués de notre modèle médical ;
- déléguer à AWS une partie de l'exploitation de la base (infrastructure, disponibilité, sauvegardes selon configuration) ;
- utiliser PyMongo et une partie des requêtes et outils déjà connus, **après validation de compatibilité** ;
- disposer d'options de dimensionnement et de disponibilité adaptées à une application de production.

**Limites majeures :**

1. DocumentDB **n'est pas le moteur MongoDB** : une compatibilité d'API, même annoncée pour la version 8.0, ne signifie pas que toutes les commandes, types, index et fonctionnalités sont identiques.
2. Notre fichier `docker/init_users.js` est un mécanisme propre au démarrage de l'image Docker `mongo:8.0` : il **ne sera pas exécuté automatiquement par DocumentDB**. Les comptes et rôles devront être provisionnés avec une procédure adaptée au service.
3. La connexion PyMongo devra être adaptée : point de terminaison privé, chiffrement TLS, certificat CA et paramètres de connexion recommandés par AWS (notamment `retryWrites=false` selon l'exemple officiel).
4. La tarification peut comprendre la capacité de calcul, le stockage, les entrées/sorties et les sauvegardes, selon les options retenues.

**Conclusion intermédiaire :** DocumentDB est la principale solution **documentaire managée native AWS** à évaluer, mais il faut effectuer un **POC de compatibilité** avant une migration de production.

**Sources :** [Compatibilité DocumentDB avec MongoDB](https://docs.aws.amazon.com/documentdb/latest/devguide/compatibility.html) · [API, opérations et types pris en charge](https://docs.aws.amazon.com/documentdb/latest/devguide/mongo-apis.html) · [Connexion PyMongo avec TLS](https://docs.aws.amazon.com/documentdb/latest/devguide/connect_programmatically.html) · [Tarifs](https://aws.amazon.com/documentdb/pricing/).

## 6. Amazon ECS — Exécution de notre conteneur Python

**Fonctionnement.** Amazon Elastic Container Service (ECS) permet d'exécuter et d'orchestrer des applications conteneurisées. Avec le mode **AWS Fargate**, AWS gère l'infrastructure sous-jacente et nous facturons principalement les ressources réservées/utilisées par les tâches selon le modèle applicable.

**Application à notre projet :**

1. construire l'image Docker du script `migration.py` ;
2. publier l'image dans **Amazon ECR**, le registre d'images d'AWS ;
3. définir une **tâche ECS Fargate** pour lancer la migration ponctuellement ou selon une planification ;
4. transmettre les paramètres de connexion et secrets de manière sécurisée ;
5. suivre son exécution et ses erreurs dans **Amazon CloudWatch**.

**Avantages :** réutilisation de notre conteneur Python, automatisation des exécutions, possibilité d'ajuster les ressources, pas de serveur EC2 à administrer en mode Fargate.

**Limites / adaptations :** ECS n'est pas une base de données. La base sera hébergée séparément. Notre `docker-compose.yml` n'est pas directement le fichier de déploiement ECS : il faut une *task definition*, une configuration réseau et la publication de l'image. Les chemins locaux de fichiers et le nom réseau `mongodb` doivent être remplacés par des ressources AWS.

**Source :** [Architecture et capacités d'ECS](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/ecs-configuration.html) · [Images et Amazon ECR](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/create-container-image.html) · [Tarification Fargate](https://aws.amazon.com/fargate/pricing/).

## 7. Alternative importante : MongoDB Atlas hébergé sur AWS

**MongoDB Atlas** est une offre de MongoDB Inc. pour exploiter une base MongoDB gérée dans le cloud, notamment sur AWS. Ce n'est **pas un service AWS natif**, mais une possibilité de conserver réellement MongoDB tout en utilisant des services AWS autour.

**Avantages :**

- meilleure continuité avec le moteur MongoDB utilisé dans nos tests locaux ;
- moins de risques de divergences fonctionnelles par rapport à un autre moteur compatible avec les API ;
- gestion managée de la base, options de sauvegarde, de réplication, et de montée en charge selon l'offre choisie ;
- possibilités de connectivité privée avec AWS (par exemple PrivateLink) selon la configuration et l'offre.

**Limites :** coût et conditions commerciales à étudier ; configuration de sécurité et des accès à réaliser ; dépendance à un autre fournisseur en plus d'AWS. Les rôles et scripts d'initialisation devront être vérifiés dans l'environnement Atlas, sans supposer qu'un `docker-entrypoint-initdb.d` s'y exécute.

**Pourquoi le présenter au client ?** Le cahier des charges demande **MongoDB**. Atlas offre une voie directe pour conserver la technologie déjà développée ; DocumentDB représente un compromis intéressant si une solution native AWS est prioritaire.

**Sources :** [MongoDB Atlas sur AWS](https://www.mongodb.com/docs/atlas/reference/amazon-aws/) · [Options de déploiement MongoDB](https://www.mongodb.com/docs/deployment/).

## 8. Comparaison des options de base de données

| Solution | Moteur MongoDB réel ? | Administration | Réutilisation de notre projet | Limite principale |
|---|---|---|---|---|
| MongoDB auto-hébergé sur EC2 | Oui | Forte (à notre charge) | Bonne, avec adaptations | Maintenance, sauvegardes, haute disponibilité et montée en charge à gérer |
| MongoDB Atlas sur AWS | Oui | Réduite (service managé) | Très bonne en principe, tests à effectuer | Coût et fournisseur supplémentaire |
| Amazon DocumentDB | Non, API compatible | Réduite (service managé AWS) | Possible, **POC indispensable** | Différences de fonctionnalités et de commandes |
| Amazon RDS | Non, base SQL | Réduite (service managé AWS) | Faible sans refonte | Modèle relationnel différent |

**Lecture du tableau :** pour notre projet déjà développé sur MongoDB, Atlas minimise le changement de technologie ; DocumentDB est l'alternative documentaire AWS native à tester ; RDS ne répond pas directement au besoin.

**Attention :** aucune option ne doit être déclarée « moins chère » ou « plus rapide » sans mesure réelle de la charge, comparaison des configurations et devis.

## 9. Proposition d'architecture AWS

### 9.1 Option privilégiée : conserver MongoDB (Atlas hébergé sur AWS)

```text
                   AWS — région européenne adaptée

      Bucket S3 privé (CSV / exports / archives)
                       |
                       v
      Tâche ECS Fargate : migration.py
      (image Python stockée dans ECR)
                       |
              connexion privée + TLS
                       v
          MongoDB Atlas hébergé sur AWS
              collection hospitalisations

      Services transversaux :
      IAM / Secrets Manager / KMS / CloudWatch / VPC
```

**Flux envisagé :** S3 conserve le fichier d'entrée ; la tâche Python lit le fichier, nettoie et transforme les lignes ; la base MongoDB stocke les documents. Les logs sont centralisés ; les identifiants sont gérés comme des secrets ; les sauvegardes de la base sont configurées et testées séparément.

### 9.2 Variante native AWS : DocumentDB

Reprendre le même principe, mais remplacer **MongoDB Atlas** par **Amazon DocumentDB**. Le script Python et les procédures utilisateurs devront être adaptés à sa compatibilité, à sa connexion TLS et à ses outils d'administration.

**Deux limites à expliciter pour notre prototype :**

- Le script actuel lit un CSV dans `data/` ; il faudra ajouter un téléchargement S3 ou une lecture S3.
- Notre Docker Compose local est une preuve de conteneurisation ; le déploiement ECS nécessite une configuration dédiée (tâches, réseau, images ECR, logs et secrets).

Cette architecture est **proposée**, pas testée sur AWS.

## 10. Sécurité et données médicales

Même si l'exercice repose sur un dataset de démonstration, une version destinée à de **véritables données personnelles de santé** impliquerait des exigences plus strictes.

**Mesures recommandées :**

- **Région et résidence des données :** sélectionner une région de l'Espace économique européen adaptée aux besoins contractuels (par exemple Paris si les services et offres choisis y sont éligibles) et contrôler également la localisation des sauvegardes.
- **Réseau privé :** placer les tâches et points d'accès à la base dans un VPC avec des règles réseau restrictives ; éviter une base publiquement accessible.
- **Secrets :** utiliser AWS Secrets Manager pour les identifiants et éventuellement leur rotation ; éviter les secrets dans Git ou dans l'image Docker.
- **Chiffrement :** TLS pour les connexions, chiffrement des stockages et, selon les exigences, clés gérées dans AWS KMS.
- **Accès :** IAM pour les services AWS, comptes/permissions de base distincts pour l'analyste, le Data Engineer et l'admin.
- **Traçabilité :** logs et alertes sans affichage d'informations médicales individuelles ; contrôle des accès.
- **Résilience :** sauvegardes, politique de conservation, tests de restauration, stratégie de reprise après incident.

**Conformité :** pour de véritables données de santé relevant du droit français, vérifier les obligations **RGPD** et éventuellement **HDS** selon la nature de l'activité. AWS indique disposer d'une certification HDS v2, mais cela **ne rend pas automatiquement conforme notre propre application**. La responsabilité du client concernant les configurations, les traitements et les contrats demeure.

**Sources :** [AWS HDS France](https://aws.amazon.com/fr/compliance/hds/) · [Responsabilité partagée AWS](https://aws.amazon.com/fr/compliance/shared-responsibility-model/) · [AWS Secrets Manager](https://docs.aws.amazon.com/secretsmanager/latest/userguide/intro.html) · [Chiffrement S3](https://docs.aws.amazon.com/AmazonS3/latest/userguide/UsingEncryption.html).

## 11. Coûts — comparaison qualitative

Le coût réel dépend de la **région**, de la **taille des ressources**, du **temps d'utilisation** et des **volumes transférés**. Il n'est pas possible d'établir un prix crédible avec la seule taille du CSV d'exercice.

| Élément | Principaux facteurs de coût |
|---|---|
| S3 | Volume stocké, classe de stockage, requêtes, transferts |
| ECS / Fargate | vCPU, mémoire et durée des tâches, stockage additionnel |
| Amazon ECR / CloudWatch | Stockage des images, logs et métriques |
| DocumentDB | Configuration de la base, calcul, stockage, I/O et sauvegardes selon l'offre |
| MongoDB Atlas | Offre/cluster, stockage, sauvegardes et transferts |
| Réseau / sécurité AWS | Transferts, NAT Gateway et éventuels points de terminaison privés, KMS/Secrets Manager |

**Méthode proposée :** créer un devis dans [AWS Pricing Calculator](https://calculator.aws/) pour deux scénarios :

1. **POC / faible usage :** import ponctuel, peu d'utilisateurs, faible volume.
2. **Production / croissance :** imports fréquents, disponibilité renforcée, sauvegardes, supervision et hausse des requêtes.

Comparer le coût mensuel estimé ainsi que les coûts d'exploitation humaine, d'administration et de migration.

**Sources tarifs :** [S3](https://aws.amazon.com/s3/pricing/) · [DocumentDB](https://aws.amazon.com/documentdb/pricing/) · [ECS](https://aws.amazon.com/ecs/pricing/) · [Fargate](https://aws.amazon.com/fargate/pricing/).

## 12. Tests nécessaires avant un choix définitif

Pour passer de l'étude à un déploiement réel, nous proposerions un **POC** (*Proof of Concept*) avec des données fictives ou autorisées.

1. **Connexion :** vérifier PyMongo, le TLS, les comptes et les accès privés.
2. **Import :** transférer un jeu d'essai, vérifier le nombre, les champs, les types et les valeurs.
3. **Fonctionnalités :** tester nos index, nos filtres sur `medical.pathologie`, les dates, les CRUD et les rôles utilisateurs.
4. **Rejouabilité :** prouver qu'une relance ne crée pas de doublons ; améliorer la règle actuelle fondée seulement sur le comptage en cas de besoin.
5. **Performance :** mesurer le débit d'import, la latence des recherches, l'usage des ressources et les coûts.
6. **Résilience :** vérifier sauvegarde, restauration, reprise après une erreur et disponibilité attendue.
7. **Conformité :** confirmer la région, les contrats et les services éligibles aux exigences réglementaires éventuelles.

**Point particulier DocumentDB :** vérifier les différences d'API et d'administration (notamment les rôles utilisateurs, les index et les options du pilote). Une compatibilité de protocole n'est pas une garantie d'équivalence fonctionnelle.

## 13. Recommandation pour DataSoluTech

### Recommandation principale : MongoDB Atlas sur AWS + ECS Fargate + S3

Cette solution paraît la plus cohérente **si l'objectif prioritaire est de conserver MongoDB** et de réutiliser notre modèle documentaire et notre code PyMongo, tout en réduisant l'administration de l'infrastructure.

- **MongoDB Atlas :** hébergement managé de la base documentaire ;
- **Amazon ECS Fargate :** exécution de notre script de migration conteneurisé ;
- **Amazon S3 :** stockage privé du CSV, des exports et des archives ;
- **ECR / CloudWatch / Secrets Manager / IAM / KMS / réseau privé :** composants nécessaires à un fonctionnement professionnel.

### Alternative : Amazon DocumentDB + ECS Fargate + S3

À privilégier si le client souhaite une base documentaire managée **native AWS** et si le POC confirme que les fonctionnalités MongoDB réellement utilisées sont compatibles.

### Pourquoi pas RDS ?

RDS est un excellent service pour les bases relationnelles, mais il demanderait ici une refonte du modèle et du script. Il n'est pas pertinent pour une migration directe MongoDB.

### Conclusion

**Il n'existe pas de solution universelle.** Notre recommandation préliminaire privilégie la continuité technologique (MongoDB Atlas sur AWS), tout en maintenant DocumentDB comme alternative AWS native. Le choix final dépendra des tests de compatibilité, des exigences de sécurité, des objectifs de disponibilité et du coût réel du scénario de production.

**Ce que nous pouvons présenter à Boris :** une comparaison documentée, une architecture cible, des risques identifiés et une méthode claire de validation avant engagement financier. Aucun déploiement AWS n'a été réalisé dans le cadre de cette étude.

## 14. Sources principales consultées

Documentation officielle vérifiée le **5 novembre 2026** :

1. [Amazon S3 — introduction](https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html)
2. [Amazon RDS — moteurs et fonctionnalités](https://aws.amazon.com/rds/features/)
3. [Amazon DocumentDB — compatibilité MongoDB](https://docs.aws.amazon.com/documentdb/latest/devguide/compatibility.html)
4. [Amazon DocumentDB — fonctionnalités prises en charge](https://docs.aws.amazon.com/documentdb/latest/devguide/mongo-apis.html)
5. [Amazon DocumentDB — connexion depuis Python](https://docs.aws.amazon.com/documentdb/latest/devguide/connect_programmatically.html)
6. [Amazon ECS — configurations et Fargate](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/ecs-configuration.html)
7. [MongoDB Atlas hébergé sur AWS](https://www.mongodb.com/docs/atlas/reference/amazon-aws/)
8. [AWS — certification HDS](https://aws.amazon.com/fr/compliance/hds/)
9. [AWS — modèle de responsabilité partagée](https://aws.amazon.com/fr/compliance/shared-responsibility-model/)
10. [AWS — calculateur de coûts](https://calculator.aws/)

> **Note de méthode :** les fonctionnalités et tarifs AWS peuvent évoluer. Les chiffres financiers devront être actualisés en fonction de la région et du dimensionnement réel avant une proposition commerciale.